# R Master v7c · Silhouette Precision Refine

基线：**v7b Lapine Silhouette Reference Transfer**。

这一版不是重新做体型，而是对 v7b 的结果做第二轮保守精修：

- 腰侧 inward curve 更顺；
- 髋外扩不再突然；
- 臀部后峰位置和体积继续向 Lapine 参考靠近；
- 臀下缘 → 大腿后侧过渡更自然；
- 大腿根内外侧过渡继续收顺；
- 保留 505 骨；
- 不改四肢骨长；
- 不烘焙 Rest Pose；
- 不导最终 VRM；
- 不改生产网页。

### 安全策略
1. 继续硬保护 UpperArm / LowerArm / Hand 顶点；
2. 若成人结构 vertex groups 存在，则硬保护；
3. 即使成人组缺失，也启用 **中央骨盆几何保护区**，避免误改关键成人结构表面；
4. 单顶点最大位移 12 mm；
5. v7c 从 v7b 预览继续，不回退到 v2。

### 输出
- 8 张标准诊断图：
  1. 全身正面
  2. 全身侧面
  3. 全身背面
  4. 全身 3/4
  5. 胸廓→腰
  6. 腰→骨盆
  7. 骨盆→大腿根
  8. 臀线侧视
- report.json
- profile_compare.csv
- review ZIP


In [ ]:
from google.colab import drive, files
from pathlib import Path
import shutil, subprocess, zipfile, json, os, csv, math

BUILD_TAG="v7c_refine_20260919_r1"

print("R Master v7c · Silhouette Precision Refine")
drive.mount("/content/drive")

ROOT=Path("/content/drive/MyDrive/R_Master")
SRC=ROOT/"v7b_silhouette"/"latest"/"R_Master_v7b_SILHOUETTE_PREVIEW.blend"
CACHE=ROOT/"cache"
OUT=ROOT/"v7c_refine"/"latest"

CACHE.mkdir(parents=True,exist_ok=True)
OUT.mkdir(parents=True,exist_ok=True)

if not SRC.exists() or SRC.stat().st_size < 50*1024*1024:
    raise RuntimeError("没找到 v7b 预览文件。先不要上传任何东西，把这一屏截图给二蛋。")

print(f"✓ v7b 源：{SRC.stat().st_size/1024/1024:.1f} MiB")
print("✓ 输出目录：MyDrive/R_Master/v7c_refine/latest/")



In [ ]:
BLENDER_VERSION="4.4.3"
BLENDER_URL="https://download.blender.org/release/Blender4.4/blender-4.4.3-linux-x64.tar.xz"

LOCAL=Path("/content/r_master_v7c")
LOCAL.mkdir(parents=True,exist_ok=True)

ARCHIVE=LOCAL/f"blender-{BLENDER_VERSION}-linux-x64.tar.xz"
BDIR=LOCAL/f"blender-{BLENDER_VERSION}-linux-x64"
DRIVE_ARCHIVE=CACHE/ARCHIVE.name

if shutil.which("xvfb-run") is None:
    subprocess.run(["apt-get","update","-qq"],check=True,stdout=subprocess.DEVNULL,stderr=subprocess.STDOUT)
    subprocess.run(
        ["apt-get","install","-y","-qq","xvfb","libgl1","libx11-6","libxi6","libxrender1","libxfixes3","libxkbcommon0","libsm6"],
        check=True,stdout=subprocess.DEVNULL,stderr=subprocess.STDOUT
    )

if DRIVE_ARCHIVE.exists() and DRIVE_ARCHIVE.stat().st_size > 100*1024*1024:
    shutil.copy2(DRIVE_ARCHIVE,ARCHIVE)
    print("✓ 复用 Blender Drive 缓存")
else:
    print("补下载 Blender 一次…")
    subprocess.run(["wget","-q","--show-progress","-O",str(ARCHIVE),BLENDER_URL],check=True)
    shutil.copy2(ARCHIVE,DRIVE_ARCHIVE)
    print("✓ Blender 已写入 Drive 缓存")

if not (BDIR/"blender").exists():
    if BDIR.exists():
        shutil.rmtree(BDIR)
    subprocess.run(["tar","-xf",str(ARCHIVE),"-C",str(LOCAL)],check=True)

BLENDER=BDIR/"blender"
if not BLENDER.exists():
    raise RuntimeError("Blender 解压失败。")

print("✓ Blender 4.4.3 就绪")



In [ ]:
BUILD_SCRIPT=LOCAL/"R_Master_v7c_Build.py"
RENDER_SCRIPT=LOCAL/"R_Master_v7c_Render.py"
BUILD_SCRIPT.write_text("\nimport bpy, os, sys, json, csv, math\nfrom mathutils import Vector\n\nargv=sys.argv[sys.argv.index(\"--\")+1:] if \"--\" in sys.argv else []\nout=None; tag=None\nfor i,a in enumerate(argv):\n    if a==\"--out\" and i+1<len(argv): out=argv[i+1]\n    if a==\"--tag\" and i+1<len(argv): tag=argv[i+1]\nif not out: raise RuntimeError(\"missing --out\")\nos.makedirs(out,exist_ok=True)\n\nbody=bpy.data.objects.get(\"R2_Mona_Main\")\nrig=bpy.data.objects.get(\"R_Master_Align_v2_PREVIEW\") or bpy.data.objects.get(\"Mona_Armature\")\nif not body or body.type!=\"MESH\": raise RuntimeError(\"R2_Mona_Main missing\")\nif not rig or rig.type!=\"ARMATURE\": raise RuntimeError(\"R Master/Mona armature missing\")\n\n# Keep only the v7b true body shell.\nremoved=[]\nfor obj in list(bpy.data.objects):\n    if obj.type==\"MESH\" and obj!=body:\n        removed.append(obj.name)\n        bpy.data.objects.remove(obj,do_unlink=True)\n\ndisabled=[]\nfor mod in body.modifiers:\n    if mod.type in {\"MASK\",\"SURFACE_DEFORM\",\"CLOTH\",\"PARTICLE_SYSTEM\"} or \"mask\" in mod.name.lower():\n        disabled.append({\"name\":mod.name,\"type\":mod.type})\n        mod.show_viewport=False\n        mod.show_render=False\n    elif mod.type in {\"MULTIRES\",\"SUBSURF\"}:\n        mod.levels=min(mod.levels,1)\n        mod.render_levels=min(mod.render_levels,1)\n\nbpy.context.view_layer.update()\nmesh=body.data\nmw=body.matrix_world.copy()\nimw=mw.inverted()\norig=[mw@v.co for v in mesh.vertices]\n\n# Lapine-derived lower-body reference profile used in v7b.\nLAPINE=[\n (-0.24,0.1295,0.0644),(-0.22,0.1308,0.0707),(-0.20,0.1367,0.0707),\n (-0.18,0.1361,0.0758),(-0.16,0.1426,0.0758),(-0.14,0.1442,0.0781),\n (-0.12,0.1459,0.0786),(-0.10,0.1459,0.0787),(-0.08,0.1460,0.0787),\n (-0.06,0.1447,0.0838),(-0.04,0.1414,0.0957),(-0.02,0.1388,0.1028),\n ( 0.00,0.1405,0.1050),( 0.02,0.1352,0.1050),( 0.04,0.1297,0.1047),\n ( 0.06,0.1251,0.1041),( 0.08,0.1201,0.1000),( 0.10,0.1137,0.0753),\n ( 0.12,0.1135,0.0753),( 0.14,0.1030,0.0590),( 0.16,0.0918,0.0461),\n ( 0.18,0.0921,0.0461)\n]\n\ndef interp_profile(dy,col):\n    if dy<=LAPINE[0][0]: return LAPINE[0][col]\n    if dy>=LAPINE[-1][0]: return LAPINE[-1][col]\n    for a,b in zip(LAPINE[:-1],LAPINE[1:]):\n        if a[0]<=dy<=b[0]:\n            t=(dy-a[0])/(b[0]-a[0])\n            return a[col]*(1-t)+b[col]*t\n    return LAPINE[-1][col]\n\ndef smoothstep(a,b,x):\n    if a==b: return 1.0 if x>=b else 0.0\n    t=max(0.0,min(1.0,(x-a)/(b-a)))\n    return t*t*(3.0-2.0*t)\n\ndef bell(x,a,b,c,d):\n    return smoothstep(a,b,x)*(1.0-smoothstep(c,d,x))\n\n# Hip anchor.\nul=rig.pose.bones.get(\"UpperLeg.L\") or rig.pose.bones.get(\"L_UpperLeg\")\nur=rig.pose.bones.get(\"UpperLeg.R\") or rig.pose.bones.get(\"R_UpperLeg\")\nif not ul or not ur: raise RuntimeError(\"UpperLeg.L/R missing\")\nhip_center=((rig.matrix_world@ul.head)+(rig.matrix_world@ur.head))*0.5\n\n# Posterior orientation.\nposterior_sign=1.0\nass=rig.pose.bones.get(\"Assbutt.L\") or rig.pose.bones.get(\"Ass.L\")\nfront=rig.pose.bones.get(\"Labia.L\") or rig.pose.bones.get(\"LowerBodyStretch\")\nif ass and front:\n    ay=(rig.matrix_world@ass.head).y\n    fy=(rig.matrix_world@front.head).y\n    posterior_sign=1.0 if ay>=fy else -1.0\n\nEDIT_GROUPS=[\"LowerBodyStretch\",\"Hip.L\",\"Hip.R\",\"UpperLeg.L\",\"UpperLeg.R\",\"Ass.L\",\"Ass.R\"]\nUPPER_PROTECT=[\"UpperArm.L\",\"UpperArm.R\",\"LowerArm.L\",\"LowerArm.R\",\"Hand.L\",\"Hand.R\"]\nADULT_PROTECT=[\n \"Labia.L\",\"Labia.R\",\"Rectum\",\"Asshole\",\"AssholeSide.L\",\"AssholeSide.R\",\n \"Genital\",\"Vagina\",\"Vagina.L\",\"Vagina.R\"\n]\n\nedit_idx=[body.vertex_groups[g].index for g in EDIT_GROUPS if g in body.vertex_groups]\nupper_idx=[body.vertex_groups[g].index for g in UPPER_PROTECT if g in body.vertex_groups]\nadult_idx=[body.vertex_groups[g].index for g in ADULT_PROTECT if g in body.vertex_groups]\n\nif len(edit_idx)<5: raise RuntimeError(\"lower-body edit vertex groups missing\")\nif len(upper_idx)<6: raise RuntimeError(\"upper-limb protection groups missing\")\n\ndef group_weights(v):\n    d={g.group:g.weight for g in v.groups}\n    edit=max([d.get(i,0.0) for i in edit_idx] or [0.0])\n    upper=max([d.get(i,0.0) for i in upper_idx] or [0.0])\n    adult=max([d.get(i,0.0) for i in adult_idx] or [0.0])\n    return edit,upper,adult\n\n# Geometric adult/pelvic safety zone.\n# This explicitly protects the central pelvis even when Mona_Main has no matching adult vertex groups.\ndef central_pelvis_safe(p):\n    dx=abs(p.x-hip_center.x)\n    dy=p.z-hip_center.z\n    rel_y=posterior_sign*(p.y-hip_center.y)\n    # central crotch / perineum / central glute cleft corridor\n    return (-0.105<=dy<=0.085) and (dx<=0.062) and (-0.095<=rel_y<=0.095)\n\nBIN_STEP=0.02\nBINS=[-0.24+i*BIN_STEP for i in range(22)]\n\ndef percentile(a,q):\n    if not a:return None\n    a=sorted(a)\n    x=(len(a)-1)*q\n    lo=int(math.floor(x)); hi=min(lo+1,len(a)-1); t=x-lo\n    return a[lo]*(1-t)+a[hi]*t\n\ndef collect_profile(points):\n    rows=[]\n    for dy0 in BINS:\n        xs=[]; posts=[]\n        for i,v in enumerate(mesh.vertices):\n            edit,upper,adult=group_weights(v)\n            p=points[i]\n            if edit<=0.05 or upper>1e-4 or adult>0.10 or central_pelvis_safe(p):\n                continue\n            dy=p.z-hip_center.z\n            if abs(dy-dy0)<=0.015:\n                xs.append(abs(p.x-hip_center.x))\n                post=posterior_sign*(p.y-hip_center.y)\n                if post>0: posts.append(post)\n        rows.append((dy0,percentile(xs,.98),percentile(posts,.98)))\n    return rows\n\nbefore=collect_profile(orig)\nbefore_map={round(r[0],4):r for r in before}\n\n# v7c is a refinement pass: smaller step than v7b.\nLATERAL_BLEND=0.34\nPOSTERIOR_BLEND=0.28\nMAX_SCALE_IN=0.90\nMAX_SCALE_OUT=1.025\nMAX_POST_IN=0.92\nMAX_POST_OUT=1.015\nSAFETY_CAP=0.012\n\nmoved=0\nsum_disp=0.0\nmax_disp=0.0\nupper_moved=0\nadult_moved=0\ncentral_safe_moved=0\nregion_counts={\"waist_curve\":0,\"hip_outer\":0,\"glute\":0,\"lower_glute_thigh\":0}\n\nfor i,v in enumerate(mesh.vertices):\n    edit,upper,adult=group_weights(v)\n    p=orig[i].copy()\n\n    if upper>1e-4 or adult>0.10 or central_pelvis_safe(p):\n        continue\n    if edit<=0.05:\n        continue\n\n    dy=p.z-hip_center.z\n    if dy<-0.255 or dy>0.195:\n        continue\n\n    gate=smoothstep(0.05,0.62,edit)\n    q=p.copy()\n\n    nearest=min(BINS,key=lambda x:abs(x-dy))\n    row=before_map.get(round(nearest,4))\n\n    # 1) Continue toward Lapine lateral target, but only by a smaller residual step.\n    if row and row[1] and row[1]>1e-5:\n        current_hw=row[1]\n        target_hw=interp_profile(dy,1)\n        ratio=target_hw/current_hw\n        ratio=max(MAX_SCALE_IN,min(MAX_SCALE_OUT,ratio))\n\n        # sculptural regional weighting\n        waist_curve=bell(dy,0.075,0.105,0.155,0.19)\n        hip_outer=bell(dy,-0.10,-0.065,0.075,0.115)\n        lower_glute_thigh=bell(dy,-0.22,-0.17,-0.105,-0.06)\n\n        region_weight=max(0.40, 0.82*hip_outer, 0.72*waist_curve, 0.72*lower_glute_thigh)\n        scale=1.0+(ratio-1.0)*LATERAL_BLEND*gate*region_weight\n        q.x=hip_center.x+(p.x-hip_center.x)*scale\n\n        if waist_curve>0: region_counts[\"waist_curve\"]+=1\n        if hip_outer>0: region_counts[\"hip_outer\"]+=1\n        if lower_glute_thigh>0: region_counts[\"lower_glute_thigh\"]+=1\n\n    # 2) Posterior refinement toward Lapine target.\n    post=posterior_sign*(p.y-hip_center.y)\n    if post>0.012 and row and row[2] and row[2]>1e-5:\n        target_post=interp_profile(dy,2)\n        ratio=target_post/row[2]\n        ratio=max(MAX_POST_IN,min(MAX_POST_OUT,ratio))\n\n        glute=bell(dy,-0.095,-0.055,0.055,0.105)\n        lower_glute=bell(dy,-0.185,-0.145,-0.075,-0.035)\n        post_weight=max(0.36,0.92*glute,0.70*lower_glute)\n\n        pscale=1.0+(ratio-1.0)*POSTERIOR_BLEND*gate*post_weight\n        new_post=post*pscale\n        q.y=hip_center.y+posterior_sign*new_post\n\n        if glute>0: region_counts[\"glute\"]+=1\n        if lower_glute>0: region_counts[\"lower_glute_thigh\"]+=1\n\n    # 3) Very mild smoothing of the lower-glute transition:\n    # only pulls posterior-most points slightly toward a local linear transition.\n    lower_band=bell(dy,-0.19,-0.16,-0.09,-0.055)\n    if lower_band>0 and post>0.045:\n        q.y += (-posterior_sign) * (0.0018*lower_band*gate)\n\n    delta=q-p\n    d=delta.length\n    if d>SAFETY_CAP:\n        delta*=SAFETY_CAP/d\n        q=p+delta\n        d=SAFETY_CAP\n\n    if d>1e-6:\n        moved+=1\n        sum_disp+=d\n        max_disp=max(max_disp,d)\n        v.co=imw@q\n\nbpy.context.view_layer.update()\nnew=[mw@v.co for v in mesh.vertices]\n\n# Invariant audits.\nfor i,v in enumerate(mesh.vertices):\n    edit,upper,adult=group_weights(v)\n    d=(new[i]-orig[i]).length\n    if upper>1e-4 and d>1e-7:\n        upper_moved+=1\n    if adult>0.10 and d>1e-7:\n        adult_moved+=1\n    if central_pelvis_safe(orig[i]) and d>1e-7:\n        central_safe_moved+=1\n\nif upper_moved:\n    raise RuntimeError(f\"UPPER_PROTECTION_FAIL: {upper_moved}\")\nif adult_idx and adult_moved:\n    raise RuntimeError(f\"ADULT_GROUP_PROTECTION_FAIL: {adult_moved}\")\nif central_safe_moved:\n    raise RuntimeError(f\"CENTRAL_PELVIS_PROTECTION_FAIL: {central_safe_moved}\")\n\nafter=collect_profile(new)\n\n# Profile error metrics.\ncompare=[]\nw_before=[]; w_after=[]; p_before=[]; p_after=[]\nfor dy,bef,aft in zip(BINS,before,after):\n    tw=interp_profile(dy,1)\n    tp=interp_profile(dy,2)\n    row={\n      \"dy_m\":dy,\n      \"lapine_half_width_m\":tw,\n      \"v7b_half_width_m\":bef[1],\n      \"v7c_half_width_m\":aft[1],\n      \"lapine_posterior_m\":tp,\n      \"v7b_posterior_m\":bef[2],\n      \"v7c_posterior_m\":aft[2]\n    }\n    compare.append(row)\n    if bef[1] is not None and aft[1] is not None:\n        w_before.append(abs(bef[1]-tw)); w_after.append(abs(aft[1]-tw))\n    if bef[2] is not None and aft[2] is not None:\n        p_before.append(abs(bef[2]-tp)); p_after.append(abs(aft[2]-tp))\n\ndef mean(a): return sum(a)/len(a) if a else None\n\ncsv_path=os.path.join(out,\"R_Master_v7c_profile_compare.csv\")\nwith open(csv_path,\"w\",newline=\"\",encoding=\"utf-8\") as f:\n    w=csv.DictWriter(f,fieldnames=list(compare[0].keys()))\n    w.writeheader(); w.writerows(compare)\n\nreport={\n \"ok\":True,\n \"stage\":\"R_Master_v7c_SilhouettePrecisionRefine\",\n \"build_tag\":tag,\n \"source_blend\":bpy.data.filepath,\n \"body_object\":body.name,\n \"rig_object\":rig.name,\n \"bone_count\":len(rig.data.bones),\n \"reference\":{\n   \"asset\":\"Lapine_BaseBody.fbx\",\n   \"archive_sha256\":\"8e215aeae8d2e42719305d66292b84c2714f616b7952591dee0383c376c86af4\",\n   \"profile_is_derived_numeric_data_only\":True\n },\n \"surface_edit\":{\n   \"moved_vertex_count\":moved,\n   \"total_vertex_count\":len(mesh.vertices),\n   \"mean_displacement_m\":sum_disp/moved if moved else 0.0,\n   \"max_displacement_m\":max_disp,\n   \"safety_cap_m\":SAFETY_CAP,\n   \"region_counts\":region_counts\n },\n \"profile_error\":{\n   \"width_mae_before_m\":mean(w_before),\n   \"width_mae_after_m\":mean(w_after),\n   \"posterior_mae_before_m\":mean(p_before),\n   \"posterior_mae_after_m\":mean(p_after)\n },\n \"protection_audit\":{\n   \"upper_limb_moved_vertex_count\":upper_moved,\n   \"adult_group_moved_vertex_count\":adult_moved,\n   \"central_pelvis_safe_moved_vertex_count\":central_safe_moved,\n   \"adult_protect_groups_present\":[g for g in ADULT_PROTECT if g in body.vertex_groups],\n   \"pass\":upper_moved==0 and adult_moved==0 and central_safe_moved==0\n },\n \"removed_mesh_count\":len(removed),\n \"disabled_modifiers\":disabled,\n \"rest_pose_baked\":False,\n \"final_vrm\":False\n}\n\nwith open(os.path.join(out,\"R_Master_v7c_report.json\"),\"w\",encoding=\"utf-8\") as f:\n    json.dump(report,f,ensure_ascii=False,indent=2)\n\nbody[\"red_master_stage\"]=\"R_Master_v7c_SilhouettePrecisionRefine\"\nbody[\"red_master_build_tag\"]=tag or \"v7c\"\nbody[\"red_master_restpose_baked\"]=False\n\nbpy.ops.wm.save_as_mainfile(\n    filepath=os.path.join(out,\"R_Master_v7c_REFINED_PREVIEW.blend\"),\n    check_existing=False\n)\n\nprint(\"[R Master v7c] BUILD_OK\")\nprint(\"[R Master v7c] moved:\",moved,\"/\",len(mesh.vertices))\nprint(\"[R Master v7c] mean/max mm:\",(sum_disp/moved*1000 if moved else 0),max_disp*1000)\nprint(\"[R Master v7c] protection:\",report[\"protection_audit\"])\nprint(\"[R Master v7c] profile error:\",report[\"profile_error\"])\n",encoding="utf-8")
RENDER_SCRIPT.write_text("\nimport bpy, os, sys\nfrom mathutils import Vector\n\nargv=sys.argv[sys.argv.index(\"--\")+1:] if \"--\" in sys.argv else []\nout=None; view=None\nfor i,a in enumerate(argv):\n    if a==\"--out\" and i+1<len(argv): out=argv[i+1]\n    if a==\"--view\" and i+1<len(argv): view=argv[i+1]\nif not out or not view: raise RuntimeError(\"missing args\")\n\nbody=bpy.data.objects.get(\"R2_Mona_Main\")\nif not body: raise RuntimeError(\"R2_Mona_Main missing\")\n\nfor obj in bpy.context.scene.objects:\n    if obj.type==\"MESH\":\n        obj.hide_render=(obj!=body)\n        obj.hide_viewport=(obj!=body)\n    elif obj.type==\"ARMATURE\":\n        obj.hide_render=True\n\nfor mod in body.modifiers:\n    if mod.type in {\"MASK\",\"SURFACE_DEFORM\",\"CLOTH\",\"PARTICLE_SYSTEM\"} or \"mask\" in mod.name.lower():\n        mod.show_viewport=False\n        mod.show_render=False\n    elif mod.type in {\"MULTIRES\",\"SUBSURF\"}:\n        mod.levels=min(mod.levels,1)\n        mod.render_levels=min(mod.render_levels,1)\n\nbody.hide_render=False\nbody.hide_viewport=False\nbpy.context.view_layer.update()\n\npts=[body.matrix_world@Vector(c) for c in body.bound_box]\nmn=Vector((min(p.x for p in pts),min(p.y for p in pts),min(p.z for p in pts)))\nmx=Vector((max(p.x for p in pts),max(p.y for p in pts),max(p.z for p in pts)))\ncenter=(mn+mx)*0.5\nheight=mx.z-mn.z\ndist=max(height,mx.x-mn.x,mx.y-mn.y)*2.55\n\nscene=bpy.context.scene\nscene.render.engine=\"BLENDER_WORKBENCH\"\nscene.render.image_settings.file_format=\"PNG\"\nscene.render.film_transparent=False\nscene.display.shading.light=\"STUDIO\"\nscene.display.shading.show_shadows=True\nscene.display.shading.show_cavity=True\nscene.display.shading.cavity_type=\"WORLD\"\nscene.display.shading.color_type=\"SINGLE\"\nscene.display.shading.single_color=(0.60,0.60,0.63)\nscene.display.shading.background_type=\"VIEWPORT\"\nscene.display.shading.background_color=(0.04,0.04,0.05)\n\ncd=bpy.data.cameras.get(\"R_v7c_Cam_DATA\") or bpy.data.cameras.new(\"R_v7c_Cam_DATA\")\ncam=bpy.data.objects.get(\"R_v7c_Cam\")\nif not cam:\n    cam=bpy.data.objects.new(\"R_v7c_Cam\",cd)\n    scene.collection.objects.link(cam)\nscene.camera=cam\ncam.data.type=\"ORTHO\"\n\ndef look_at(target):\n    cam.rotation_euler=(Vector(target)-cam.location).to_track_quat(\"-Z\",\"Y\").to_euler()\n\ndef render_file(fname,pos,target,scale,res):\n    scene.render.resolution_x,scene.render.resolution_y=res\n    scene.render.resolution_percentage=100\n    cam.location=Vector(pos)\n    cam.data.ortho_scale=scale\n    look_at(target)\n    path=os.path.join(out,fname)\n    scene.render.filepath=path\n    bpy.ops.render.render(write_still=True)\n    return path\n\nspec={\n \"front\":(\"R_Master_v7c_front.png\",(center.x,center.y-dist,center.z),center,height*1.08,(720,960)),\n \"side\":(\"R_Master_v7c_side.png\",(center.x+dist,center.y,center.z),center,height*1.08,(720,960)),\n \"back\":(\"R_Master_v7c_back.png\",(center.x,center.y+dist,center.z),center,height*1.08,(720,960)),\n \"three_quarter\":(\"R_Master_v7c_three_quarter.png\",(center.x+dist*.72,center.y-dist*.72,center.z),center,height*1.08,(720,960)),\n}\n\nfor key,frac,scale in [\n    (\"torso_waist\",.635,max(.43,height*.255)),\n    (\"waist_pelvis\",.545,max(.45,height*.265)),\n    (\"pelvis_upperthigh\",.465,max(.47,height*.275))\n]:\n    z=mn.z+height*frac\n    spec[key]=(f\"R_Master_v7c_{key}.png\",(center.x,center.y-dist,z),(center.x,center.y,z),scale,(900,700))\n\nz=mn.z+height*.515\nspec[\"glute_side\"]=(\n    \"R_Master_v7c_glute_side.png\",\n    (center.x+dist,center.y,z),\n    (center.x,center.y,z),\n    max(.56,height*.33),\n    (900,700)\n)\n\nif view not in spec: raise RuntimeError(\"unknown view \"+str(view))\nfname,pos,target,scale,res=spec[view]\nprint(\"[R Master v7c] RENDER_OK\",view,render_file(fname,pos,target,scale,res))\n",encoding="utf-8")

STAGE=OUT/"R_Master_v7c_REFINED_PREVIEW.blend"
REPORT=OUT/"R_Master_v7c_report.json"

rebuild=True
if STAGE.exists() and STAGE.stat().st_size>50*1024*1024 and REPORT.exists():
    try:
        rebuild=json.loads(REPORT.read_text(encoding="utf-8")).get("build_tag")!=BUILD_TAG
    except:
        rebuild=True

if rebuild:
    TMP=LOCAL/"stage"
    if TMP.exists(): shutil.rmtree(TMP)
    TMP.mkdir(parents=True,exist_ok=True)
    tlog=TMP/"R_Master_v7c_build.log"
    cmd=["xvfb-run","-a",str(BLENDER),"--background",str(SRC),"--python",str(BUILD_SCRIPT),"--","--out",str(TMP),"--tag",BUILD_TAG]
    with tlog.open("w",encoding="utf-8") as log:
        p=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in p.stdout:
            log.write(line)
            if "R Master v7c" in line or "Traceback" in line or "Error" in line:
                print(line.rstrip())
        rc=p.wait()

    if rc!=0:
        print(tlog.read_text(encoding="utf-8",errors="replace")[-12000:])
        raise RuntimeError(f"v7c 构建失败，退出码 {rc}。截图给二蛋即可。")

    for name in [
        "R_Master_v7c_REFINED_PREVIEW.blend",
        "R_Master_v7c_report.json",
        "R_Master_v7c_profile_compare.csv",
        "R_Master_v7c_build.log"
    ]:
        src=TMP/name
        if not src.exists():
            raise RuntimeError("v7c 构建缺少："+name)
        shutil.copy2(src,OUT/name)
    print("✓ v7c 已写入 Drive")
else:
    print("✓ v7c 已存在，跳过重建")

r=json.loads(REPORT.read_text(encoding="utf-8"))
print("  骨骼数：",r["bone_count"])
print("  修改顶点：",r["surface_edit"]["moved_vertex_count"],"/",r["surface_edit"]["total_vertex_count"])
print("  平均位移：",round(r["surface_edit"]["mean_displacement_m"]*1000,2),"mm")
print("  最大位移：",round(r["surface_edit"]["max_displacement_m"]*1000,2),"mm")
print("  保护验收：",r["protection_audit"])
print("  轮廓误差：",r["profile_error"])



In [ ]:
views=[
 ("front","R_Master_v7c_front.png"),
 ("side","R_Master_v7c_side.png"),
 ("back","R_Master_v7c_back.png"),
 ("three_quarter","R_Master_v7c_three_quarter.png"),
 ("torso_waist","R_Master_v7c_torso_waist.png"),
 ("waist_pelvis","R_Master_v7c_waist_pelvis.png"),
 ("pelvis_upperthigh","R_Master_v7c_pelvis_upperthigh.png"),
 ("glute_side","R_Master_v7c_glute_side.png"),
]

print("④ 断点渲染 8 视图…")
for i,(view,fname) in enumerate(views,1):
    dest=OUT/fname
    if dest.exists() and dest.stat().st_size>20_000:
        print(f"✓ [{i}/8] {view} 已存在，跳过")
        continue

    print(f"▶ [{i}/8] {view}…")
    cmd=["xvfb-run","-a",str(BLENDER),"--background",str(STAGE),"--python",str(RENDER_SCRIPT),"--","--out",str(OUT),"--view",view]
    p=subprocess.run(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
    important=[x for x in p.stdout.splitlines() if "R Master v7c" in x or "Traceback" in x or "Error" in x]
    if important:
        print("\n".join(important[-10:]))
    if p.returncode!=0:
        raise RuntimeError(f"{view} 渲染失败，退出码 {p.returncode}")
    if not dest.exists():
        raise RuntimeError(f"{view} 未生成")
    print(f"✓ [{i}/8] 已写入 Drive")

print("✓ v7c 八张图完成")



In [ ]:
from IPython.display import display,Image,Markdown

items=[
 ("全身正面","R_Master_v7c_front.png"),
 ("全身侧面","R_Master_v7c_side.png"),
 ("全身背面","R_Master_v7c_back.png"),
 ("全身 3/4","R_Master_v7c_three_quarter.png"),
 ("胸廓→腰","R_Master_v7c_torso_waist.png"),
 ("腰→骨盆","R_Master_v7c_waist_pelvis.png"),
 ("骨盆→大腿根","R_Master_v7c_pelvis_upperthigh.png"),
 ("臀线侧视","R_Master_v7c_glute_side.png"),
]

for title,f in items:
    p=OUT/f
    if p.exists():
        display(Markdown(f"### {title}"))
        display(Image(filename=str(p),width=500))

zpath=OUT/"R_Master_v7c_Review.zip"
if zpath.exists():
    zpath.unlink()

with zipfile.ZipFile(zpath,"w",compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for _,f in items:
        p=OUT/f
        if p.exists():
            z.write(p,arcname=f)
    for f in ["R_Master_v7c_report.json","R_Master_v7c_profile_compare.csv","R_Master_v7c_build.log"]:
        p=OUT/f
        if p.exists():
            z.write(p,arcname=f)

print(f"✓ Review ZIP：{zpath.stat().st_size/1024/1024:.1f} MiB")
print("重 .blend 留在 Drive，不下载到手机。")
files.download(str(zpath))

